In [2]:
import numpy as np
import matplotlib.pyplot as plt
import glob

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Flatten, Dense
from tensorflow.keras.applications.resnet50 import ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [3]:
IMAGE_SIZE = [224,224]
train_Folder = '../archive/alien_vs_predator_thumbnails/data/train/'
test_Folder = '../archive/alien_vs_predator_thumbnails/data/validation/'

In [4]:
myResNet_def = ResNet50(include_top=False, weights='imagenet', input_shape=IMAGE_SIZE + [3])
# print(myResNet_def.summary())

In [5]:
for layer in myResNet_def.layers:
    layer.trainable = False

In [6]:
classes = glob.glob(train_Folder + '/*')
print(classes)
numOfClasses = len(classes)

['../archive/alien_vs_predator_thumbnails/data/train\\alien', '../archive/alien_vs_predator_thumbnails/data/train\\predator']


In [7]:
global_avg_pooling_layer = tf.keras.layers.GlobalAveragePooling2D()(myResNet_def.output)
flatten_layer = Flatten()(global_avg_pooling_layer)
pred_layer = Dense(numOfClasses, activation='softmax')(flatten_layer)

In [8]:
model = Model(inputs = myResNet_def.input, outputs = pred_layer)
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 224, 224, 3  0           []                               
                                )]                                                                
                                                                                                  
 conv1_pad (ZeroPadding2D)      (None, 230, 230, 3)  0           ['input_1[0][0]']                
                                                                                                  
 conv1_conv (Conv2D)            (None, 112, 112, 64  9472        ['conv1_pad[0][0]']              
                                )                                                                 
                                                                                              

In [9]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=.001),
    loss='categorical_crossentropy',
    metrics=['accuracy'])

In [10]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=.2,
    zoom_range=.2,
    horizontal_flip=True,
    vertical_flip=True,
    rotation_range=25,
)

test_datagen = ImageDataGenerator(
    rescale=1./255,
)

In [11]:
train_set = train_datagen.flow_from_directory(
    train_Folder,
    target_size=(224,224),
    batch_size=24,
    class_mode='categorical'
)

test_set = test_datagen.flow_from_directory(
    test_Folder,
    target_size=IMAGE_SIZE,
    batch_size=24,
    class_mode='categorical'
)

Found 694 images belonging to 2 classes.
Found 200 images belonging to 2 classes.


In [12]:
EPOCHS = 100
best_model = '../archive/alien_vs_predator_thumbnails/best_model.h5'

In [13]:
from keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping

In [14]:
callback = [
    ModelCheckpoint(best_model, verbose=1, save_best_only=True, monitor='val_accuracy'),
    ReduceLROnPlateau(monitor='val_accuracy', patience=10, verbose=1, min_lr=1e-4),
    EarlyStopping(monitor='val_accuracy', patience=30, verbose=1)
           ]

In [15]:
model_run = model.fit(
    train_set,
    validation_data=test_set,
    epochs=EPOCHS,
    steps_per_epoch=len(train_set),
    validation_steps=len(test_set),
    callbacks=callback
)

Epoch 1/100
29/29 [==============================] - ETA: 0s - loss: 0.7349 - accuracy: 0.4986
Epoch 1: val_accuracy improved from -inf to 0.50500, saving model to ../archive/alien_vs_predator_thumbnails\best_model.h5
29/29 [==============================] - 89s 3s/step - loss: 0.7349 - accuracy: 0.4986 - val_loss: 0.6919 - val_accuracy: 0.5050 - lr: 0.0010
Epoch 2/100
29/29 [==============================] - ETA: 0s - loss: 0.6654 - accuracy: 0.5576
Epoch 2: val_accuracy improved from 0.50500 to 0.61500, saving model to ../archive/alien_vs_predator_thumbnails\best_model.h5
29/29 [==============================] - 82s 3s/step - loss: 0.6654 - accuracy: 0.5576 - val_loss: 0.6446 - val_accuracy: 0.6150 - lr: 0.0010
Epoch 3/100
29/29 [==============================] - ETA: 0s - loss: 0.6558 - accuracy: 0.6138
Epoch 3: val_accuracy did not improve from 0.61500
29/29 [==============================] - 82s 3s/step - loss: 0.6558 - accuracy: 0.6138 - val_loss: 0.6413 - val_accuracy: 0.6150 - 

In [33]:
import cv2
import os
import numpy as np
from keras.utils import load_img, img_to_array

In [47]:
def prepairImg(pathToImg, Image_Size = 224):
    img = load_img(pathToImg, target_size=(Image_Size, Image_Size))
    img_res = img_to_array(img)
    img_res = np.expand_dims(img_res, axis = 0)
    img_res = img_res/255.
    return img_res

In [39]:
Image_Size = 224
CLASSES = os.listdir(train_Folder)
num_classes = len(CLASSES)
print(num_classes)

2


In [63]:
load_model = tf.keras.models.load_model(best_model)

In [49]:
alien_test = '../archive/alien_ModelTest.png'
pred_test = '../archive/predator modelTest.png'

In [59]:
alien = cv2.imread(alien_test)
pred =  cv2.imread(pred_test)

In [57]:
alien_prep = prepairImg(alien_test)
pred_prep = prepairImg(pred_test)

In [69]:
results_alien = load_model.predict(alien_prep, verbose = 1)
results_pred = load_model.predict(pred_prep, verbose = 1)

answer_1 = np.argmax(results_alien, axis = 1)
answer_2 = np.argmax(results_pred, axis = 1)

index_1 = answer_1[0]
index_2 = answer_2[0]

print(index_1, index_2)

1/1 [==============================] - 0s 122ms/step
0 1


In [71]:
CLASSES[index_1], CLASSES[index_2]

('alien', 'predator')